# Sentiment analysis
Using sample selection based on similarity to train a neural net to predict sentiment.

## Data
The data used for this project come from Amazon reviews ([source](https://nijianmo.github.io/amazon/index.html))

In [11]:
# download datasets
# !wget -P ../data http://deepyeti.ucsd.edu/jianmo/amazon/categoryFilesSmall/Digital_Music_5.json.gz

# !wget -P ../data http://deepyeti.ucsd.edu/jianmo/amazon/categoryFilesSmall/Video_Games_5.json.gz

# !wget -P ../data http://deepyeti.ucsd.edu/jianmo/amazon/categoryFilesSmall/Arts_Crafts_and_Sewing_5.json.gz

### Data pre processing
- read json.gz file
- Remove 3 star reviews
- map sentiment to star reviews (1, 2 = negative and 4,5 = positive)
- concatenate review title and review text
- select only relevant columns (concat review and sentiment)
- remove duplicates
- pickle dataframe 

In [ ]:
# import utils

# utils.pre_process('../data/Digital_Music_5.json.gz', 'music')
# utils.pre_process('../data/Video_Games_5.json.gz', 'games')
# utils.pre_process('../data/Arts_Crafts_and_Sewing_5.json.gz', 'art')

### Compare domains
Concat all reviews into a big string, and compare the strings to find the cosine similarity.

tfidf score of a word w is `tf(w)*idf(w)`  
Where, tf(w) = Number of times the word appears in a document/Total number of words in the document
and idf(w) = Number of documents/Number of documents that contains word w ([source](https://kanoki.org/2018/12/27/text-matching-cosine-similarity/))

In [13]:
def make_big_string(df, n):
    '''
    input: dataframe with reviews and number of reviews to compare
    output: all values in the column concatenated
    '''
    # n is the number of reviews we want to compare
    big_string = ' '.join(df.iloc[:n,0].astype(str))
    return big_string.lower()


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def cos_sim_df(str1, str2):
    '''
    input: 2 strings
    output: cosine similarity and dataframe with tfidf score for each word
    '''
    corpus = [str1, str2]

    # tokanise -> remove strop words, select only words (ignore punctuation, digits, etc)
    vectorizer = TfidfVectorizer(stop_words='english', token_pattern='[a-z]\w+')
    trsfm = vectorizer.fit_transform(corpus)

    return cosine_similarity(trsfm[0:1], trsfm)[0][1], pd.DataFrame(trsfm.toarray(),columns=vectorizer.get_feature_names_out(),index=['str1','str2'])

In [2]:
import pandas as pd

In [3]:
df_music = pd.read_pickle('../data/pickled_dfs/df_music.pkl')  
df_games = pd.read_pickle('../data/pickled_dfs/df_games.pkl')  
df_art = pd.read_pickle('../data/pickled_dfs/df_art.pkl')  

In [4]:
len(df_music), len(df_games), len(df_art)

(95623, 364935, 339612)

In [14]:
cs_music_games, df_music_games = cos_sim_df(make_big_string(df_music, 95623), make_big_string(df_games, 364935))
cs_music_games

0.326263528198846

In [15]:
cs_music_art, df_music_art = cos_sim_df(make_big_string(df_music, 95623), make_big_string(df_art, 339612))
cs_music_art

0.5038188814870144

## Model training

Train CNN for sentiment analysis

### Split datasets
Into train, val and test sets.

In [7]:
from sklearn.model_selection import train_test_split

def split_dataset(df, random_state=42):
    X, y = df['rev_sum'], df['sentiment']

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, train_size = 0.8, stratify=y, random_state=42)
    X_dev, X_test, y_dev, y_test = train_test_split(X_temp, y_temp, test_size = 0.5, train_size = 0.5, stratify=y_temp, random_state=42)

    return X_train, y_train, X_dev, y_dev, X_test, y_test

In [8]:
X_train, y_train, X_dev, y_dev, X_test, y_test = split_dataset(df_music)

### Tokenize the data
Tokenize the reviews to get the embeddings to feed the neural net

In [12]:
import sklearn

In [ ]:
def tokenize(s):
    return sklearn.utils.(stop_words='english', token_pattern='[a-z]\w+')
TEXT = data.Field(tokenize=tokenize)
LABEL = data.LabelField()

In [38]:
X_train[:3].astype(str)

95493     Five Stars Every time I here this song it just...
148945    favorite song My favorite version of this song...
23785     So Catchy, and Great Summer Song I love this s...
Name: rev_sum, dtype: object

In [42]:
# from: https://machinelearningmastery.com/prepare-text-data-machine-learning-scikit-learn/

from sklearn.feature_extraction.text import TfidfVectorizer
# list of text documents
l = list(X_train[:5].astype(str))
text = list(map(str.lower, l))

# create the transform
vectorizer = TfidfVectorizer(stop_words='english', token_pattern='[a-z]\w+')
# tokenize and build vocab
vectorizer.fit(text)

# summarize
print(vectorizer.vocabulary_)
print(vectorizer.idf_)

# encode document
vector = vectorizer.transform([text[0]])

# summarize encoded vector
print(vector.shape)
print(vector.toarray())

{'stars': 55, 'time': 57, 'song': 54, 'just': 24, 'inspires': 23, 'walk': 63, 'faith': 14, 'favorite': 15, 'version': 60, 'live': 28, 'red': 45, 'rock': 49, 'arizona': 2, 'catchy': 7, 'great': 18, 'summer': 56, 'love': 31, 'vibe': 61, 'reminds': 48, 'hot': 21, 'night': 40, 'longing': 29, 'ocean': 42, 'sexy': 52, 'vocals': 62, 'guitar': 19, 'melody': 36, 'took': 58, 'needed': 39, 'really': 44, 'feel': 16, 'loved': 32, 'lord': 30, 'like': 25, 'accepted': 0, 'need': 38, 'buy': 6, 'enjoy': 13, 'nbsp': 37, 'data': 10, 'hook': 20, 'product': 43, 'link': 26, 'linked': 27, 'class': 8, 'normal': 41, 'href': 22, 'setlist': 51, 'best': 5, 'donnie': 11, 'mcclurkin': 35, 'dp': 12, 'b0063r96oa': 4, 'ref': 46, 'cm_cr_arp_d_rvw_txt': 9, 'utf8': 59, 'selena': 50, 'gomez': 17, 'amazing': 1, 'artist': 3, 'lyrics': 33, 'simple': 53, 'remember': 47, 'makes': 34}
[2.09861229 2.09861229 2.09861229 2.09861229 2.09861229 2.09861229
 2.09861229 1.69314718 2.09861229 2.09861229 2.09861229 2.09861229
 2.09861229 

In [ ]:
# use this to make CNN https://towardsdatascience.com/sentiment-classification-using-cnn-in-pytorch-fba3c6840430